In [4]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import os
import json
from copy import deepcopy
from dotenv import load_dotenv
from datetime import datetime, timezone, timedelta

load_dotenv()

True

In [ ]:
def get_area_weather_data(pageNo=1, numOfRows=10):
    """
    기상청 TAF API를 호출하여 인천공항(RKSI)의 최신 예보를 받아온 후,
    향후 24시간 동안의 시간별 날씨/기온 데이터프레임을 생성하여 반환합니다.
    """
    
    # --- 1. API 호출 및 기본 파싱 ---
    url = 'https://apihub.kma.go.kr/api/typ02/openApi/AmmIwxxmService/getTaf'
    params ={'authKey' : os.environ['id'], 'pageNo' : str(pageNo), 'numOfRows' : str(numOfRows), 'dataType' : 'JSON', 'icao' : 'RKSI'}

    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # HTTP 에러 발생 시 예외 처리
        data = response.json()
        xml_string = data['response']['body']['items']['item'][0]['tafMsg']
    except (requests.exceptions.RequestException, KeyError, IndexError) as e:
        print(f"API 데이터를 가져오는 데 실패했습니다: {e}")
        return pd.DataFrame() # 실패 시 빈 데이터프레임 반환

    # --- 2. XML 파싱 및 네임스페이스 정의 ---
    root = ET.fromstring(xml_string)
    namespaces = {
        'iwxxm': 'http://icao.int/iwxxm/2.0',
        'aixm': 'http://www.aixm.aero/schema/5.1.1',
        'gml': 'http://www.opengis.net/gml/3.2',
        'om': 'http://www.opengis.net/om/2.0',
        'xlink': 'http://www.w3.org/1999/xlink'
    }

    # --- 3. (추가) 기온 정보 추출 ---
    try:
        temp_forecast = root.find(".//iwxxm:temperature/iwxxm:AerodromeAirTemperatureForecast", namespaces)
        min_temp = float(temp_forecast.find("iwxxm:minimumAirTemperature", namespaces).text)
        max_temp = float(temp_forecast.find("iwxxm:maximumAirTemperature", namespaces).text)
        min_temp_time = pd.to_datetime(temp_forecast.find(".//iwxxm:minimumAirTemperatureTime//gml:timePosition", namespaces).text)
        max_temp_time = pd.to_datetime(temp_forecast.find(".//iwxxm:maximumAirTemperatureTime//gml:timePosition", namespaces).text)
        temp_series = pd.Series({min_temp_time: min_temp, max_temp_time: max_temp})
    except AttributeError:
        print("XML에서 기온 정보를 찾을 수 없습니다.")
        return pd.DataFrame()


    # --- 4. 예보 구간별 데이터 리스트 생성 ---
    forecast_list = []
    base_forecast_element = root.find(".//iwxxm:baseForecast", namespaces)
    valid_time_element = root.find(".//iwxxm:validTime/gml:TimePeriod", namespaces)

    current_conditions = {
        "예보구분": "기본 예보",
        "적용시작시간": valid_time_element.find("gml:beginPosition", namespaces).text,
        "적용종료시간": valid_time_element.find("gml:endPosition", namespaces).text,
        "바람방향(도)": base_forecast_element.find('.//iwxxm:meanWindDirection', namespaces).text,
        "바람속도(노트)": base_forecast_element.find('.//iwxxm:meanWindSpeed', namespaces).text,
        "시정(m)": base_forecast_element.find('.//iwxxm:prevailingVisibility', namespaces).text if base_forecast_element.find('.//iwxxm:prevailingVisibility', namespaces) is not None else '9999',
        "특이사항": "없음"
    }
    forecast_list.append(current_conditions)

    for change_fcst in root.findall(".//iwxxm:changeForecast", namespaces):
        new_forecast = deepcopy(current_conditions)
        record = change_fcst.find(".//iwxxm:MeteorologicalAerodromeForecastRecord", namespaces)
        phenomenon_time = change_fcst.find(".//om:phenomenonTime/gml:TimePeriod", namespaces)
        
        change_indicator = record.get('changeIndicator')
        if change_indicator == "TEMPORARY_FLUCTUATIONS": new_forecast["예보구분"] = "일시적 변화 (TEMPO)"
        elif change_indicator == "BECOMING": new_forecast["예보구분"] = "점진적 변화 (BECMG)"
        else: new_forecast["예보구분"] = change_indicator

        new_forecast["적용시작시간"] = phenomenon_time.find("gml:beginPosition", namespaces).text
        new_forecast["적용종료시간"] = phenomenon_time.find("gml:endPosition", namespaces).text
        
        vis = record.find("iwxxm:prevailingVisibility", namespaces)
        if vis is not None: new_forecast["시정(m)"] = vis.text
            
        weather = record.find("iwxxm:weather", namespaces)
        if weather is not None:
            href_value = weather.get(f'{{{namespaces["xlink"]}}}href')
            if href_value: new_forecast["특이사항"] = href_value.split('/')[-1]
            else: new_forecast["특이사항"] = "없음"
        else: new_forecast["특이사항"] = "없음"

        forecast_list.append(new_forecast)
        
        if change_indicator == "BECOMING":
            current_conditions = new_forecast

            
    # --- 5. 시간별 데이터프레임 생성 및 가공 ---
    df = pd.DataFrame(forecast_list)
    df['적용시작시간'] = pd.to_datetime(df['적용시작시간'])
    forecast_starts = df.set_index('적용시작시간')[['바람방향(도)', '바람속도(노트)', '시정(m)']]

    now = pd.Timestamp.now(tz='UTC').floor('h')
    hourly_index = pd.date_range(start=now, periods=24, freq='h')

    combined_df = pd.concat([forecast_starts, pd.DataFrame(index=hourly_index)])
    combined_df['기온'] = temp_series
    combined_df = combined_df.sort_index()

    combined_df['기온'].interpolate(method='time', inplace=True)
    combined_df.ffill(inplace=True)
    combined_df.bfill(inplace=True)

    hourly_df = combined_df.loc[hourly_index].copy()
    hourly_df.reset_index(inplace=True)
    
    # 시간대 변환 및 날짜/시간 컬럼 분리
    hourly_df['시간대'] = hourly_df['index'].dt.tz_convert('Asia/Seoul')
    hourly_df['날짜'] = hourly_df['시간대'].dt.strftime('%Y-%m-%d')
    hourly_df['시간'] = hourly_df['시간대'].dt.strftime('%H')
    
    hourly_df.rename(columns={'바람방향(도)': '바람방향', '바람속도(노트)': '바람속도', '시정(m)': '시정'}, inplace=True)
    
    final_cols = ['날짜', '시간', '기온', '바람방향', '바람속도', '시정']
    hourly_df = hourly_df[final_cols]
    hourly_df['기온'] = hourly_df['기온'].round(1)
    hourly_df = hourly_df.astype({'바람방향': 'int', '바람속도': 'int', '시정': 'int'})

    return hourly_df

In [6]:
hourly_df = get_area_weather_data(1, 10)
print(hourly_df)

C:\Users\LSS\AppData\Local\Temp\ipykernel_4120\910376428.py:100: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined_df['기온'].interpolate(method='time', inplace=True)


            날짜  시간  기온  바람방향  바람속도    시정
0   2025-07-24  18 NaN   330     6  9999
1   2025-07-24  19 NaN   330     6  9999
2   2025-07-24  20 NaN   330     6  9999
3   2025-07-24  21 NaN   330     6  9999
4   2025-07-24  22 NaN   330     6  9999
5   2025-07-24  23 NaN   330     6  9999
6   2025-07-25  00 NaN   330     6  9999
7   2025-07-25  01 NaN   330     6  9999
8   2025-07-25  02 NaN   330     6  9999
9   2025-07-25  03 NaN   330     6  9999
10  2025-07-25  04 NaN   330     6  9999
11  2025-07-25  05 NaN   330     6  9999
12  2025-07-25  06 NaN   330     6  9999
13  2025-07-25  07 NaN   330     6  9999
14  2025-07-25  08 NaN   330     6  9999
15  2025-07-25  09 NaN   330     6  9999
16  2025-07-25  10 NaN   330     6  9999
17  2025-07-25  11 NaN   330     6  9999
18  2025-07-25  12 NaN   330     6  9999
19  2025-07-25  13 NaN   330     6  9999
20  2025-07-25  14 NaN   330     6  9999
21  2025-07-25  15 NaN   330     6  9999
22  2025-07-25  16 NaN   330     6  9999
23  2025-07-25  

In [ ]:
import psycopg2
from psycopg2.extras import execute_values

# 1. PostgreSQL 연결(커넥션 유지)
conn = psycopg2.connect(
    host='host',
    database='postgres',
    user='user',
    password='password',
    port=5432
)

cursor = conn.cursor()

In [8]:
table_name = 'silver.api_silver_hourly_weather_forecast'
print(f"'{table_name}' 테이블의 모든 데이터를 삭제합니다...")
cursor.execute(f"DELETE FROM {table_name};")

# 데이터프레임을 튜플의 리스트 형태로 변환
tuples = [tuple(x) for x in hourly_df.to_numpy()]

# 삽입할 컬럼들을 문자열로 만듦
# hourly_df의 컬럼 이름을 DB 테이블에 맞게 변경 (예시)
hourly_df.columns = ['date', 'hour', 'temperature', 'wind_direction', 'wind_speed', 'visibility']

# 데이터프레임의 컬럼명으로 cols 문자열을 동적으로 생성
cols = ','.join(list(hourly_df.columns))
    
# INSERT 쿼리 템플릿 생성
query = f"INSERT INTO {table_name} ({cols}) VALUES %s"

# execute_values를 사용해 데이터 삽입
execute_values(cursor, query, tuples)
    
# 변경 사항을 데이터베이스에 커밋
conn.commit()
    
print(f"'{table_name}' 테이블에 {len(hourly_df)}개의 행이 성공적으로 삽입되었습니다.")

'silver.api_silver_hourly_weather_forecast' 테이블의 모든 데이터를 삭제합니다...
'silver.api_silver_hourly_weather_forecast' 테이블에 24개의 행이 성공적으로 삽입되었습니다.


In [9]:
cursor.close()
conn.close()